## 3CX CALLING REPORT

In [140]:
import pandas as pd
import numpy as np
import gspread
from gspread_pandas import Spread, conf
from datetime import datetime
import pytz

In [141]:
# Use read_csv for .csv files
df = pd.read_csv(r"C:\Users\USER\Downloads\Emarath_global\3CX_Calling_Report_Dataset\call_reports.csv")

In [142]:
df['username'] = df['From'].str.split(' \\(').str[0]
df_filtered = df[df['username'].str.lower().str.endswith('global', na=False)]
df_filtered.head(2)

,Call Time,Call ID,From,To,Direction,Status,Ringing,Talking,Cost,Call Activity Details,Sentiment,Summary,Transcription,username
1,2026-03-28T15:48:19,00000000-01dc-be9c-330e-2083000042d0,Rahiyad Emarathglobal (136),0097433178528,Outbound,Unanswered,00:00:01,00:00:00,0.0,Dialed: (0097433178528) → Via rule: Outbound ...,NaN,NaN,NaN,Rahiyad Emarathglobal
2,2026-03-28T15:48:11,00000000-01dc-be9c-2e73-bd52000042cf,Adwaitha Emarathglobal (134),00966565473059,Outbound,Unanswered,00:00:03,00:00:00,0.0,Dialed: (00966565473059) → Via rule: Outbound...,NaN,NaN,NaN,Adwaitha Emarathglobal


In [143]:
df_filtered['Call Time'] = pd.to_datetime(df_filtered['Call Time'])
df_filtered['Date'] = df_filtered['Call Time'].dt.date

In [144]:
df_filtered['dialed count'] = df_filtered.groupby(['Call ID', 'From'])['To'].transform('nunique')
df_new = df_filtered[[ 'From','To', 'dialed count', 'Status', 'Direction','Call Activity Details']].copy()
df_summary = df_new

In [145]:
# Wrong number (Indian Number)
df_filtered['To'] = df_filtered['To'].astype(str).str.replace('+', '').str.strip()
is_indian = df_filtered['To'].str.startswith('91') | df_filtered['To'].str.startswith('0091')
indian_numbers_df = df_filtered[is_indian]
indian_count = len(indian_numbers_df)

print(f"Total Indian numbers found: {indian_count}")
indian_numbers_df[['Call Time', 'To', 'username']].head(2)

Total Indian numbers found: 0


,Call Time,To,username


In [146]:
status_counts = df_filtered.groupby('From')['Status'].value_counts().unstack(fill_value=0)
status_counts = status_counts[['Answered', 'Unanswered']]  
status_counts['Dailed Count'] = status_counts['Answered'] + status_counts['Unanswered']
# status_counts.head(2)

In [147]:
df_filtered['Talking_seconds'] = pd.to_timedelta(df_filtered['Talking']).dt.total_seconds()
total_duration = df_filtered.groupby('From')['Talking_seconds'].sum().reset_index()
total_duration['Total Duration (minutes)'] = total_duration['Talking_seconds'] / 60
total_duration = total_duration.rename(columns={'From': 'User'})[['User', 'Total Duration (minutes)']]
total_duration['Total Duration (minutes)']=total_duration['Total Duration (minutes)'].round(2)

In [148]:
total_seconds = df_filtered.groupby('From')['Talking_seconds'].sum()
answered_calls = status_counts['Answered']
average_duration_minutes = (total_seconds / 60).div(answered_calls).replace([float('inf'), -float('inf')], 0).fillna(0).round(2)
avg_df = average_duration_minutes.reset_index()
avg_df.columns = ['User', 'Avg Minutes per Answered Call']

In [149]:
status_counts_reset = status_counts.reset_index()
merged_df = status_counts_reset.merge(total_duration, left_on='From', right_on='User', how='left').drop('User', axis=1)
merged_df = merged_df.merge(avg_df, left_on='From', right_on='User', how='left').drop('User', axis=1)

In [150]:
BDM1_List = ['Chaithanya Emarathglobal (127)','SHAMNA Emarathglobal (132)','Burhana Emarathglobal (119)','Adwaitha Emarathglobal (134)','Habiya Emarathglobal (122)', 'Rahib Emarathglobal (125)']
BDM2_List = ['Amina Emarathglobal (120)','Ranjith Emarathglobal (119)','Rahiyad Emarathglobal (136)','Akash Emarathglobal (123)','Neha Emarathglobal (128)','Afnan Emarathglobal (130)']
BDM3_List = ['Nihad Emarathglobal (129)','Arshad Emarathglobal (121)','Safwan K P Emarathglobal (137)','Zakiya Emarathglobal (138)','Shihad Emarathglobal (133)', 'Najiya Emarathglobal (124)']
BDM1_df = merged_df[merged_df['From'].isin(BDM1_List)]
BDM2_df = merged_df[merged_df['From'].isin(BDM2_List)]
BDM3_df = merged_df[merged_df['From'].isin(BDM3_List)]

In [151]:
names_to_filter = [ "Vaishakh Emarathglobal (126)", "Ihsan Emarathglobal (139)"]
logi_call_report_df = merged_df[merged_df['From'].isin(names_to_filter)]

In [152]:
blacklist = [
    "Vaishakh Emarathglobal (126)",
    'AJITH T R Emarathglobal (140)',
    "Ihsan Emarathglobal (139)"
]

merged_df = merged_df[~merged_df['From'].isin(blacklist)]
merged_df = merged_df[~merged_df['From'].str.lower().isin([x.lower() for x in blacklist])]

col_to_move = merged_df.pop('Dailed Count')
merged_df.insert(1, 'Dailed Count', col_to_move)

In [153]:
col_to_move = logi_call_report_df.pop('Dailed Count')
logi_call_report_df.insert(1, 'Dailed Count', col_to_move)
logi_call_report_df.head(3)

,From,Dailed Count,Answered,Unanswered,Total Duration (minutes),Avg Minutes per Answered Call
13,Vaishakh Emarathglobal (126),56,12,44,4.68,0.39


In [154]:
col_to_move = BDM1_df.pop('Dailed Count')
BDM1_df.insert(1, 'Dailed Count', col_to_move)
col_to_move = BDM2_df.pop('Dailed Count')
BDM2_df.insert(1, 'Dailed Count', col_to_move)
col_to_move = BDM3_df.pop('Dailed Count')
BDM3_df.insert(1, 'Dailed Count', col_to_move)
# BDM1_df.head(2)

In [155]:
merged_df.rename(columns={'From': 'BDE'}, inplace=True)
merged_df.head(2)

,BDE,Dailed Count,Answered,Unanswered,Total Duration (minutes),Avg Minutes per Answered Call
0,Adwaitha Emarathglobal (134),144,29,115,39.80,1.37
1,Afnan Emarathglobal (130),148,30,118,53.58,1.79


In [156]:
config_path = r'C:\Users\USER\Downloads\Emarath_global\service_account.json'
c = conf.get_config(file_name=config_path)


report_sheet_url = "https://docs.google.com/spreadsheets/d/1PqMNtFU0bas_BGYFhIYWojO2petW-TOGxeRB2OWnF08/edit?gid=113745152#gid=113745152"

PENDING_SHEET_URL = "https://docs.google.com/spreadsheets/d/1EPap1JYLO0exlKpDReT47LFnDZM4TBx_O19sXLbEVz8/edit?gid=1430538233#gid=1430538233" 
PENDING_SPREAD = Spread(PENDING_SHEET_URL, config=c)
spread = Spread(report_sheet_url, config=c)

In [ ]:



report_date = datetime.now()
# yesterday = datetime.now() - timedelta(days=1)
date_str = report_date.strftime("%d-%m-%y")

# Save to Excel
file_path = f"./Output_Data/user_call_summary_{date_str}.xlsx"
merged_df.to_excel(file_path, index=False)
print(f"Data saved to user_call_summary_{date_str}.xlsx")

dynamic_tab_name = f"3cx_report_{date_str}"
print(f"The report will be saved to tab: {dynamic_tab_name}")

spread.open_sheet(dynamic_tab_name, create=True)
PENDING_SPREAD.open_sheet("AGENT CALL REPORT", create=True)
spread.df_to_sheet(merged_df, index=False, replace=True, start="A3")
PENDING_SPREAD.df_to_sheet(merged_df, index=False, replace=True, start="A2")
print(f"Data successfully pushed to tab: {dynamic_tab_name}")

Data saved to user_call_summary_28-03-26.xlsx
The report will be saved to tab: 3cx_report_28-03-26
Data successfully pushed to tab: 3cx_report_28-03-26


In [158]:

tab_name = "logistic_call_report"
logi_report_date = datetime.now().strftime("%Y-%m-%d")

if 'Report Date' not in logi_call_report_df.columns:
    logi_call_report_df.insert(0, 'Report Date', logi_report_date)

try:
    logi_spread = Spread(report_sheet_url, config=c)
    logi_spread.open_sheet(tab_name, create=False)

    existing_df = logi_spread.sheet_to_df()
    is_empty = existing_df.empty
    
    current_data_rows = len(existing_df)
    start_row = 1 if is_empty else current_data_rows + 2

    logi_spread.df_to_sheet(
        logi_call_report_df, 
        index=False, 
        replace=False, 
        headers=is_empty,
        start=f'A{start_row}' 
    )

    print(f"Data successfully appended to '{tab_name}' starting at row {start_row}.")

except Exception as e:
    print(f"Detailed Error: {e}")

Data successfully appended to 'logistic_call_report' starting at row 17.


In [159]:
BDM_sheet_url = "https://docs.google.com/spreadsheets/d/19B1Vclt9U0AOu-yk99E960g5tYRQ9JRX-M2GlAPvE6U/edit#gid=0"

tabs_to_process = {
    "BDM1": BDM1_df,
    "BDM2": BDM2_df,
    "BDM3": BDM3_df
}


ist = pytz.timezone('Asia/Kolkata')
now_ist = datetime.now(ist)
call_report_date = now_ist.strftime("%Y-%m-%d")
call_report_time = now_ist.strftime("%H:%M")


try:
    bdm_spread = Spread(BDM_sheet_url, config=c)
    
    for tab_name, df in tabs_to_process.items():
        cols_to_add = {'Report Date': call_report_date, 'Report Time': call_report_time}
        
        for idx, (col_name, value) in enumerate(cols_to_add.items()):
            if col_name not in df.columns:
                df.insert(idx, col_name, value)
            else:
                df[col_name] = value

        bdm_spread.open_sheet(tab_name, create=True)
        PENDING_SPREAD.open_sheet(tab_name, create=True)
        bdm_spread.sheet.batch_clear(['A2:Z10000']) 
        PENDING_SPREAD.sheet.batch_clear(['A2:Z10000'])
        upload_df = df.astype(str)
        bdm_spread.df_to_sheet(upload_df, index=False, headers=False, start="A2")
        PENDING_SPREAD.df_to_sheet(upload_df, index=False, headers=False, start="A2")
        
        print(f"✅ {tab_name} refreshed for {call_report_date}.")

    print(f"Success: All {len(tabs_to_process)} tabs updated.")

except Exception as e:
    print(f"❌ Detailed Error: {e}")

✅ BDM1 refreshed for 2026-03-28.
✅ BDM2 refreshed for 2026-03-28.
✅ BDM3 refreshed for 2026-03-28.
Success: All 3 tabs updated.
